In [1]:
import os
import sys
import torch
from pathlib import Path
from itertools import chain
from functools import partial

if "__file__" in globals():
    project_root = Path(__file__).resolve().parent.parent
else:
    project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

project_root = project_root.resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

if Path.cwd() != project_root:
    os.chdir(project_root)

from torch.utils.data import random_split, DataLoader
from core.config import load_config, print_config
from core.data.loader import setup_dataset, load_finetuning_dataset
from core.data.dataset import finetuning_collate_fn, FinetuningDataset
from core.data.transforms import BeatmapTransform, BeatmapNormalizer
from core.model.bert import BertForContrastiveFineTuning
from core.training.sampler import create_contrastive_sampler
from core.training.finetuner import setup_finetuning
from core.training.checkpoint import CheckpointManager
from core.logger import print_data_summary

In [2]:
print(f"PyTorch version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Working directory: {os.getcwd()}")

config = load_config("config", config_dir=".")
print_config(config, "Loaded Fine-Tuning Configuration")

PyTorch version: 2.8.0+cu126
Using device: cuda
Working directory: /home/jessiez/osu_corpora

--- Loaded Fine-Tuning Configuration ---
data:
  db_path: ./data/beatmap_dataset/
  max_seq_len: 1023
  val_split: 0.1
  max_samples_per_class:
    aim: 1500
training:
  batch_size: 8
  num_epochs: 5
  learning_rate: 0.0005
  weight_decay: 0.05
  warmup_ratio: 0.05
  min_lr: 1.0e-06
  use_amp: true
  checkpoint_dir: ./checkpoints
  grad_clip_norm: 1.0
  gradient_accumulation_steps: 4
  sampling:
    method: kde
    kde_bandwidth: 0.5
    num_bins: 200
    expand_for_augmentation: true
contrastive:
  user_tag_classes: 42
  collection_label_classes: 5
  temperature: 0.1
  num_positives_per_anchor: 1
  user_tag_weight: 0.0
  collection_label_weight: 1.0
  difficulty_rating_weight: 1.0
  num_epochs: 5
model:
  d_model: 512
  n_heads: 8
  n_layers: 6
  dim_feedforward_mult: 4
  dropout: 0.1
components:
  use_flash_attention: true
  compile_model: true
  compile_mode: default
mlm:
  masking_ratio: 0

In [3]:
colab_url = 'https://drive.google.com/uc?id=14yvshmHQ069SCjKIa8uBak8aPcyccSqv'
DATASET_PATH = setup_dataset(config['data']['db_path'], colab_url)

print(f"Using database: {DATASET_PATH}")

all_beatmaps_data, difficulty_ratings, all_labels, all_tags = load_finetuning_dataset(
    DATASET_PATH,
    max_seq_len=config['data']['max_seq_len'],
    max_samples_per_class=config['data'].get('max_samples_per_class')
)

print_data_summary(all_beatmaps_data)
print(f"Loaded {len(all_labels)} label entries and {len(all_tags)} tag entries.")

Using database: ./data/beatmap_dataset/
Loading fine-tuning dataset with labels and tags (memory-efficiently)...
Loading labels from ./data/labels.json...
Applying max samples per class limit...
Downsampling class 'aim' from 2711 to 1500 samples.
Found 5884 unique beatmaps for fine-tuning. Loading only this subset...
Loading raw data from Parquet dataset...
Filtering dataset to 5884 specific beatmap IDs before processing.
Loaded 4202 beatmaps and 4586049 hit objects.
Engineering features for all beatmaps (vectorized)...


/home/jessiez/osu_corpora/core/data/loader.py:45: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = df.groupby('beatmap_id')


Converting processed dataframes to tensors...


100%|██████████| 4201/4201 [00:00<00:00, 432845.41it/s]


Applying log transforms and filtering by sequence length...


100%|██████████| 4201/4201 [00:00<00:00, 11773.89it/s]

Finished loading and processing all data.
Assembling final labels and tags...
Final fine-tuning dataset size: 4201 beatmaps.
Finished loading fine-tuning dataset.

--- Data Summary ---
Total beatmaps: 4201
Vector dimension: 15
Metadata dimension: 6
Sequence length - Min: 69, Max: 1023, Avg: 873.7
--------------------
Loaded 4201 label entries and 4201 tag entries.


In [4]:
all_unique_labels = sorted(list(set(chain.from_iterable(all_labels))))
collection_label_encoder = {label: i for i, label in enumerate(all_unique_labels)}

all_unique_tags = sorted(list(set(chain.from_iterable(all_tags))))
user_tag_encoder = {tag: i for i, tag in enumerate(all_unique_tags)}

config['contrastive']['collection_label_classes'] = len(all_unique_labels)
config['contrastive']['user_tag_classes'] = len(all_unique_tags) if user_tag_encoder else 0

print(f"Found {len(collection_label_encoder)} unique collection labels.")
print(f"Found {len(user_tag_encoder)} unique user tags.")

val_size = int(len(all_beatmaps_data) * config['data']['val_split'])
train_size = len(all_beatmaps_data) - val_size
indices = list(range(len(all_beatmaps_data)))
train_indices, val_indices = random_split(indices, [train_size, val_size])

print(f"Data split: {len(train_indices)} training, {len(val_indices)} validation")

train_subset_data = [all_beatmaps_data[i] for i in train_indices]
val_subset_data = [all_beatmaps_data[i] for i in val_indices]
train_ratings = [difficulty_ratings[i] for i in train_indices]
val_ratings = [difficulty_ratings[i] for i in val_indices]
train_labels = [all_labels[i] for i in train_indices]
val_labels = [all_labels[i] for i in val_indices]
train_tags = [all_tags[i] for i in train_indices]
val_tags = [all_tags[i] for i in val_indices]

pretrain_checkpoint_manager = CheckpointManager(
    config['training']['checkpoint_dir'],
    model_name=config['model'].get('type', 'model')
)

print("Attempting to load normalization stats from pre-trained checkpoint...")
stats = pretrain_checkpoint_manager.load_normalization_stats()

if stats is None:
    raise FileNotFoundError(
        "Could not load normalization stats from pre-trained checkpoint. "
    )

vector_stats, meta_stats = stats
normalizer = BeatmapNormalizer(vector_stats=vector_stats, meta_stats=meta_stats)
print("Successfully created normalizer from pre-trained stats.")

Found 5 unique collection labels.
Found 0 unique user tags.
Data split: 3781 training, 420 validation
Attempting to load normalization stats from pre-trained checkpoint...
Successfully created normalizer from pre-trained stats.


In [5]:
train_transform = BeatmapTransform(normalizer, augment=True)
val_transform = BeatmapTransform(normalizer, augment=False)

train_dataset = FinetuningDataset(train_subset_data, train_ratings, train_labels, train_tags, train_transform)
val_dataset = FinetuningDataset(val_subset_data, val_ratings, val_labels, val_tags, val_transform)

actual_vector_dim = train_subset_data[0][0].shape[1]
collate_with_args = partial(
    finetuning_collate_fn,
    max_seq_len=config['data']['max_seq_len'],
    vector_dim=actual_vector_dim,
    device=device
)

contrastive_sampler = create_contrastive_sampler(
    labels=train_labels,
    batch_size=config['training']['batch_size'],
    num_positives_per_anchor=config['contrastive']['num_positives_per_anchor']
)

train_dataloader = DataLoader(
    train_dataset,
    batch_sampler=contrastive_sampler,
    collate_fn=collate_with_args
)
val_dataloader = DataLoader(
    val_dataset,
    batch_size=config['training']['batch_size'],
    shuffle=False,
    collate_fn=collate_with_args
)

print(f"Created dataloaders with batch size: {config['training']['batch_size']}")
sample_batch = next(iter(train_dataloader))
print(f"Sample batch shapes: vectors={sample_batch[0].shape}, mask={sample_batch[1].shape}, meta={sample_batch[2].shape}, ratings={sample_batch[3].shape}")

ContrastiveBatchSampler: Found 5 labels with >=2 members.
Total data points that can be anchors: 3781
Created dataloaders with batch size: 8


/home/jessiez/osu_corpora/.venv/lib/python3.12/site-packages/torch/utils/data/sampler.py:68: UserWarning: `data_source` argument is not used and will be removed in 2.2.0.You may still have custom implementation that utilizes it.
  warnings.warn(


Sample batch shapes: vectors=torch.Size([8, 1023, 15]), mask=torch.Size([8, 1023]), meta=torch.Size([8, 6]), ratings=torch.Size([8])


In [6]:
model = BertForContrastiveFineTuning.from_config(config, device)

pretrain_checkpoint_manager = CheckpointManager(
    config['training']['checkpoint_dir'],
    model_name=config['model'].get('type', 'model')
)

if pretrain_checkpoint_manager.checkpoint_exists():
    print("Found pre-trained checkpoint. Loading BERT backbone weights...")
    pretrain_checkpoint = torch.load(
        pretrain_checkpoint_manager.get_checkpoint_path(),
        map_location=device,
        weights_only=False
    )
    
    pretrain_state_dict = pretrain_checkpoint['model_state_dict']
    
    compiled_prefix = '_orig_mod.'
    is_compiled = any(k.startswith(compiled_prefix) for k in pretrain_state_dict.keys())
    
    if is_compiled:
        pretrain_state_dict = {k[len(compiled_prefix):]: v for k, v in pretrain_state_dict.items()}

    bert_state_dict = {k.replace('bert.', ''): v for k, v in pretrain_state_dict.items() if k.startswith('bert.')}
    
    missing, unexpected = model.bert.load_state_dict(bert_state_dict, strict=False)
    print(f"Loaded BERT backbone. Missing keys: {len(missing)}, Unexpected keys: {len(unexpected)}")
else:
    print("WARNING: No pre-trained checkpoint found. Fine-tuning from scratch.")

summary = model.get_summary()
print(f"Model Summary: {summary['total_parameters'] / 1e6:.2f}M parameters")

Compiling Contrastive BERT model with torch.compile...
Found pre-trained checkpoint. Loading BERT backbone weights...
Loaded BERT backbone. Missing keys: 0, Unexpected keys: 0
Model Summary: 25.44M parameters


In [7]:
trainer, checkpoint_manager = setup_finetuning(
    model, train_dataloader, val_dataloader, config, device, normalizer,
    user_tag_encoder, collection_label_encoder
)

start_epoch = 0
if checkpoint_manager.checkpoint_exists():
    try:
        start_epoch, metrics, _, _ = checkpoint_manager.load_checkpoint(
            model, trainer.optimizer, trainer.scheduler, trainer.scaler, device=device
        )
        start_epoch += 1
        print(f"Loaded fine-tuning checkpoint, resuming from epoch {start_epoch + 1}")
        print(f"Previous validation metrics: {metrics}")
    except Exception as e:
        print(f"Could not load fine-tuning checkpoint: {e}. Starting from scratch.")

print(f"Fine-tuning setup complete. Starting from epoch {start_epoch + 1}")

Scheduler: WSD with 59 warmup, 0 stable, 1126 decay steps.
Cooldown type: cosine, Min LR Ratio: 0.0020
FineTuningTrainer initialized - AMP: True, Device: cuda, Grad Accum: 4
Effective batch size: 32
Fine-tuning setup complete. Starting from epoch 1


In [8]:
print("--- Starting BERT Fine-Tuning ---")
print(f"Model: {config['model']['n_layers']} layers, {config['model']['d_model']} dimensions")
print(f"Training on {len(train_dataset)} samples, validating on {len(val_dataset)} samples.")
print(f"Total epochs: {config['contrastive']['num_epochs']}")

metrics_tracker = trainer.train(start_epoch)

print("\n--- BERT Fine-Tuning Completed! ---")

--- Starting BERT Fine-Tuning ---
Model: 6 layers, 512 dimensions
Training on 3781 samples, validating on 420 samples.
Total epochs: 5
Starting fine-tuning from epoch 1/5...


Epoch 1 [Train]:   0%|          | 0/237 [00:00<?, ?it/s]

W0925 19:46:21.689000 513607 .venv/lib/python3.12/site-packages/torch/_inductor/utils.py:1436] [4/0_1] Not enough SMs to use max_autotune_gemm mode


Validation:   0%|          | 0/53 [00:00<?, ?it/s]

Epoch 1/5 | Time: 113.63s | Train Loss: 4.4213 | Val Loss: 0.4946 | Checkpoint saved to ./checkpoints/model_finetuned_latest.pth


...

Validation:   0%|          | 0/53 [00:00<?, ?it/s]

Epoch 4/5 | Time: 91.58s | Train Loss: 0.2496 | Val Loss: 0.3118 | Checkpoint saved to ./checkpoints/model_finetuned_latest.pth


Epoch 5 [Train]:   0%|          | 0/237 [00:00<?, ?it/s]

Validation:   0%|          | 0/53 [00:00<?, ?it/s]

Epoch 5/5 | Time: 90.41s | Train Loss: 0.1561 | Val Loss: 0.2528 | Checkpoint saved to ./checkpoints/model_finetuned_latest.pth
Fine-tuning finished.

--- BERT Fine-Tuning Completed! ---
